In [ ]:
import pandas as pd

import numpy as np
import os

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import torch
torch.manual_seed(100)

import random
random.seed(15)

import numpy as np
np.random.seed(30)

In [ ]:
with torch.no_grad():
    torch.cuda.empty_cache()

In [ ]:
DIR_INPUTS = './data_raw/'
DIR_RUNTIME_DATA = './data_runtime_surf/'
DIR_RUNTIME_RESULTS = './results_runtime_surf/'

MAX_EPOCH_TRAIN = 1000
LR_FACTOR = 0.5

torch.set_float32_matmul_precision('medium')

### Create train-validation and test sets (into csvs)

In [ ]:
df_features = pd.read_csv(os.path.join(DIR_INPUTS,'df_features.csv'), index_col=0)
df_features.index = df_features.index.rename('subject')
df_features = df_features.reset_index()
df_surfs = pd.read_csv(os.path.join(DIR_INPUTS,'df_surfs.csv'),  index_col=0)


In [ ]:
MAX_STATES = df_surfs['g'].max()
N_TIME = 11

In [ ]:
N_TIME

In [ ]:
MAX_STATES

In [ ]:
df_features.head()

In [ ]:
df_surfs.head()

In [ ]:
subj_train = df_features['subject']


for suffix, subj in [
    ('train', subj_train),
]:
    print(f'n_subj in {suffix}:{len(subj)}')
    df_features_sub = df_features.loc[
        df_features['subject'].isin(subj),
        :
    ]
    df_features_sub.to_csv(os.path.join(DIR_RUNTIME_DATA, f'df_features_{suffix}.csv'))

    df_surfs_sub = df_surfs.loc[
        df_surfs['subj'].isin(subj),
        :
    ]
    df_surfs_sub.to_csv(os.path.join(DIR_RUNTIME_DATA, f'df_surfs_{suffix}.csv'))

del df_surfs_sub
del df_features_sub
del df_surfs
del df_features

## Load train val and test sets

In [ ]:
from torch.utils.data import DataLoader
from monotonic_nn_surv_surf.utils.datasets_def import DatasetFeatANDsurf

In [ ]:
ds_train = DatasetFeatANDsurf(
    path_feat_by_subj=os.path.join(DIR_RUNTIME_DATA,'df_features_train.csv'),
    path_surf=os.path.join(DIR_RUNTIME_DATA,'df_surfs_train.csv'),
    max_grade=MAX_STATES, 
    max_time=N_TIME,
)
loader_train = DataLoader(ds_train, batch_size=1000,shuffle=True)

In [ ]:
len(ds_train)

## Train model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
import pytorch_lightning as pl

pl.seed_everything(seed=20)

In [ ]:
from monotonic_nn_surv_surf.utils.pl_model_wrapper import LitSurvSurfReg
from monotonic_nn_surv_surf.utils.surv_surf_latent import SurvSurfLatent, LatentFeatFC

### read best hyperparams and tune lr if necessary

In [ ]:
import json


with open(os.path.join(DIR_RUNTIME_RESULTS,"best_hyper_params.json"), 'r') as openfile:

    # Reading from json file
    best_hyper_params = json.load(openfile)

n_feat_neurons = best_hyper_params['n_feat_neurons']
n_monoton_neurons = best_hyper_params['n_monoton_neurons']
n_monotone_layers = best_hyper_params['n_monotone_layers']
n_feat_layers = best_hyper_params['n_feat_layers']
p_dropout = best_hyper_params['p_dropout']
learning_rate = best_hyper_params['learning_rate']*LR_FACTOR


In [ ]:
best_hyper_params

In [ ]:
model = SurvSurfLatent(
    mono_net_sizes=[n_feat_neurons] + [n_monoton_neurons]*n_monotone_layers + [1],
    latent_feat_transformer=LatentFeatFC(
        input_size=3, 
        output_size=n_feat_neurons, 
        neurons_per_layer=(n_feat_layers-1)*[n_feat_neurons],
        dropout_p=p_dropout
    ),
)

model_lit = LitSurvSurfReg(model=model, lr=learning_rate, print_epoch=True)

### actual train

In [ ]:
logger = pl.loggers.CSVLogger(save_dir=DIR_RUNTIME_RESULTS)
early_stop = pl.callbacks.EarlyStopping(monitor='val_loss', patience=20,)
trainer = pl.Trainer(
    accelerator="gpu", 
    devices=1, 
    max_epochs=MAX_EPOCH_TRAIN, 
    logger=logger,
    enable_progress_bar=False,
    default_root_dir=DIR_RUNTIME_RESULTS,
    check_val_every_n_epoch=1,
    callbacks=[early_stop]
)
trainer.fit(
    model=model_lit, 
    train_dataloaders=loader_train ,
    val_dataloaders=loader_train
)

In [ ]:
trainer.test(model=model_lit, dataloaders=loader_train)

## Inspect learning curve

In [ ]:
dir_logs = os.path.join(DIR_RUNTIME_RESULTS, 'lightning_logs')
dir_logs

In [ ]:
latest_ver = sorted(os.listdir(dir_logs), key=lambda x: int(x.split('_')[-1]))[-1]
latest_ver

In [ ]:
epoch_metrics = pd.read_csv(os.path.join(dir_logs, f'{latest_ver}/metrics.csv'))
epoch_metrics.head(20)

In [ ]:
epoch_metrics = epoch_metrics.groupby('epoch').apply(
    lambda df: 
    pd.Series(
        [
            df['val_loss'].iloc[0],
            df['train_loss'].iloc[-1]
        ],
        index=['val_loss','train_loss']
    ), 
).reset_index()

In [ ]:
epoch_metrics['val_loss'].min()

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(
    epoch_metrics['epoch'],
    epoch_metrics['train_loss'],
    label='train_loss'
)
ax.legend()

In [ ]:
with torch.no_grad():
    torch.cuda.empty_cache()